### **IMPORT DATA**

In [4]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 150)

FILE = 'bedtime_screentime_sleep_debt_kotor.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

mentah = pd.read_csv(FILE)
print('Data mentah:', mentah.shape)

Saving bedtime_screentime_sleep_debt_kotor.csv to bedtime_screentime_sleep_debt_kotor.csv
Data mentah: (8600, 18)


In [5]:
#Mengamankan data mentah
df = mentah.copy()

jejak = []
def catat(langkah, terdampak, alasan):
    """Setiap perubahan dicatat SAAT dikerjakan, bukan belakangan."""
    jejak.append({'langkah': langkah, 'baris_terdampak': int(terdampak), 'alasan': alasan})
    print(f'  {langkah}: {int(terdampak)} baris')

print('Siap. Data mentah tetap utuh di variabel `mentah`.')

Siap. Data mentah tetap utuh di variabel `mentah`.


### **Pengecekan**

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8600 entries, 0 to 8599
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8600 non-null   object 
 1   age                       8600 non-null   object 
 2   gender                    8600 non-null   object 
 3   occupation_type           8429 non-null   object 
 4   chronotype                8473 non-null   object 
 5   bedtime_phone_minutes     8600 non-null   object 
 6   primary_bedtime_app       8600 non-null   object 
 7   screen_brightness_pct     8429 non-null   float64
 8   blue_light_filter_active  8600 non-null   object 
 9   caffeine_post_5pm_mg      8429 non-null   float64
 10  physical_activity_min     8600 non-null   int64  
 11  sleep_latency_min         8344 non-null   float64
 12  total_sleep_hours         8600 non-null   float64
 13  deep_sleep_pct            8343 non-null   float64
 14  rem_slee

In [7]:
# Cek bentuk data
df.head(5)

,user_id,age,gender,occupation_type,chronotype,bedtime_phone_minutes,primary_bedtime_app,screen_brightness_pct,blue_light_filter_active,caffeine_post_5pm_mg,physical_activity_min,sleep_latency_min,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,morning_alarm_snoozes,next_day_fatigue_score,sleep_debt_category
0,USR-00001,29,Female,Healthcare / Shift Worker,Intermediate,179,INSTAGRAM / REDDIT,54.0,1,0.0,21,84.4,3.20,23.5,22.2,7,10.0,Severe Sleep Debt
1,USR-00002,58,Female,Student,Intermediate,163,YouTube,76.0,1,92.0,72,86.7,4.06,18.0,21.5,7,10.0,Severe Sleep Debt
2,USR-00003,41,Female,Healthcare / Shift Worker,Night Owl,100,TikTok / Reels,59.0,0,45.0,23,70.5,3.20,18.1,20.5,7,10.0,Severe Sleep Debt
3,USR-00004,36,Non-Binary,Remote Tech,Intermediate,34,YouTube,52.0,0,97.0,49,38.1,6.99,21.6,22.4,1,1.7,Mild Deficit
4,USR-00005,23,Female,Healthcare / Shift Worker,Intermediate,27,TikTok / Reels,82.0,1,0.0,38,32.3,4.58,24.1,NaN,4,5.3,Moderate Debt


In [8]:
# Melakukan penyeragaman data
kolom_num = ['age', 'bedtime_phone_minutes', 'screen_brightness_pct', 'caffeine_post_5pm_mg',
             'physical_activity_min', 'sleep_latency_min', 'total_sleep_hours',
             'deep_sleep_pct', 'rem_sleep_pct', 'morning_alarm_snoozes', 'next_day_fatigue_score']

PLACEHOLDER = {'n/a', 'na', '-', 'nan', 'none', 'null', ''}

# CEK: sel mana yang bukan angka murni?
for k in kolom_num:
    s = df[k].astype('string').str.strip()
    bukan_angka = s.notna() & pd.to_numeric(s, errors='coerce').isna()
    if bukan_angka.any():
        print(f'{k:25s} {bukan_angka.sum():4d} sel teks -> contoh:', s[bukan_angka].unique()[:6])

age                         31 sel teks -> contoh: <StringArray>
['45 tahun', '38 tahun', '28 tahun', '56 tahun', '42 tahun', '22 tahun']
Length: 6, dtype: string
bedtime_phone_minutes       51 sel teks -> contoh: <StringArray>
['41 min', '87 min', '17 min', '64 min', '94 min', '73 min']
Length: 6, dtype: string
rem_sleep_pct               25 sel teks -> contoh: <StringArray>
['-']
Length: 1, dtype: string


In [9]:
# Tindak lanjut penyeragaman
ditangani = {}
for k in kolom_num:
    s = df[k].astype('string').str.strip()
    awal_bukan_angka = (s.notna() & pd.to_numeric(s, errors='coerce').isna()).sum()

    s = s.mask(s.str.lower().isin(PLACEHOLDER))                  # placeholder -> NaN
    s = s.str.extract(r'(-?\d+(?:\.\d+)?)')[0]                   # ambil angkanya saja
    df[k] = pd.to_numeric(s, errors='coerce')                     # jadi float
    ditangani[k] = int(awal_bukan_angka)

catat('Ekstrak angka & placeholder -> NaN', sum(ditangani.values()),
      'Regex ambil angka dari teks bersatuan; N/A, -, NaN dijadikan missing')
print({k: v for k, v in ditangani.items() if v})
print(df[kolom_num].dtypes)

  Ekstrak angka & placeholder -> NaN: 107 baris
{'age': 31, 'bedtime_phone_minutes': 51, 'rem_sleep_pct': 25}
age                         Int64
bedtime_phone_minutes       Int64
screen_brightness_pct     Float64
caffeine_post_5pm_mg      Float64
physical_activity_min       Int64
sleep_latency_min         Float64
total_sleep_hours         Float64
deep_sleep_pct            Float64
rem_sleep_pct             Float64
morning_alarm_snoozes       Int64
next_day_fatigue_score    Float64
dtype: object


In [10]:
# Pengecekan kembali
col = df['bedtime_phone_minutes'].astype(str)
print(col.str.replace(r'[-\d.]+', '', regex=True).str.strip().value_counts())

bedtime_phone_minutes
    8600
Name: count, dtype: int64


In [11]:
# Ubah data dan tipe
df['bedtime_phone_minutes'] = (
    df['bedtime_phone_minutes']
    .astype(str)
    .str.extract(r'(-?\d+\.?\d*)')[0]
    .astype(float)
)

print(df['bedtime_phone_minutes'].dtype)
print(df['bedtime_phone_minutes'].isna().sum(), 'NaN')
print(df['bedtime_phone_minutes'].describe())

float64
0 NaN
count    8600.000000
mean       59.204186
std        36.868346
min         1.000000
25%        31.000000
50%        51.000000
75%        79.000000
max       180.000000
Name: bedtime_phone_minutes, dtype: float64


In [12]:
# Cek type data : apakah sudah benar?
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8600 entries, 0 to 8599
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8600 non-null   object 
 1   age                       8600 non-null   Int64  
 2   gender                    8600 non-null   object 
 3   occupation_type           8429 non-null   object 
 4   chronotype                8473 non-null   object 
 5   bedtime_phone_minutes     8600 non-null   float64
 6   primary_bedtime_app       8600 non-null   object 
 7   screen_brightness_pct     8429 non-null   Float64
 8   blue_light_filter_active  8600 non-null   object 
 9   caffeine_post_5pm_mg      8429 non-null   Float64
 10  physical_activity_min     8600 non-null   Int64  
 11  sleep_latency_min         8344 non-null   Float64
 12  total_sleep_hours         8600 non-null   Float64
 13  deep_sleep_pct            8343 non-null   Float64
 14  rem_slee

### Teks, placeholder, kategori

In [13]:
# CONTOH 4.1 — buang spasi pada SELURUH kolom teks sekaligus
kolom_teks = [c for c in df.columns if df[c].dtype == object or str(df[c].dtype) == 'str']

n_spasi = 0
for kol in kolom_teks:
    s = df[kol].astype('string')
    n_spasi += int((s != s.str.strip()).sum())
    df[kol] = s.str.strip()

catat('Buang spasi awal dan akhir kolom teks', n_spasi,
      'Spasi tersembunyi memecah groupby dan melambungkan nunique')

  Buang spasi awal dan akhir kolom teks: 500 baris


In [14]:
# Melakukan pengecekan variasi nilai
kolom_kat = ['gender', 'occupation_type', 'chronotype', 'primary_bedtime_app']

def kunci(s):
    """Bentuk normal untuk membandingkan: tanpa spasi tepi, huruf kecil, pemisah seragam."""
    return (s.astype('string').str.strip().str.lower()
             .str.replace(r'[\s_\-]+', ' ', regex=True))

for k in kolom_kat:
    print(f'\n=== {k} ===')
    print('Nilai unik mentah :', df[k].nunique(dropna=True))
    print('Nilai unik ternormalisasi:', kunci(df[k]).nunique(dropna=True))
    print('Missing:', df[k].isna().sum())

    # Tampilkan varian mentah yang dikelompokkan per bentuk normalnya
    varian = (df[[k]].assign(_kunci=kunci(df[k]))
                    .dropna()
                    .groupby('_kunci')[k]
                    .agg(lambda x: sorted(x.unique().tolist())))
    print(varian.to_string())


=== gender ===
Nilai unik mentah : 12
Nilai unik ternormalisasi: 5
Missing: 0
_kunci
f                                                          [F]
female                                [FEMALE, Female, female]
m                                                          [M]
male                                        [MALE, Male, male]
non binary    [NON-BINARY, Non Binary, Non-Binary, non-binary]

=== occupation_type ===
Nilai unik mentah : 15
Nilai unik ternormalisasi: 5
Missing: 171
_kunci
corporate 9 to 5             [CORPORATE 9-TO-5, Corporate 9-to-5, corporate...
freelance / creative         [FREELANCE / CREATIVE, Freelance / Creative, f...
healthcare / shift worker    [HEALTHCARE / SHIFT WORKER, Healthcare / Shift...
remote tech                            [REMOTE TECH, Remote Tech, remote tech]
student                                            [STUDENT, Student, student]

=== chronotype ===
Nilai unik mentah : 8
Nilai unik ternormalisasi: 3
Missing: 127
_kunci
intermediate    

In [15]:
# Penyeragaman konsistensi penulisan pada gender, occupation-type, chronotype, primary_bedtime_app
LABEL_BAKU = {
    'gender'             : ['Female', 'Male', 'Non-Binary'],
    'occupation_type'    : ['Corporate 9-to-5', 'Student', 'Remote Tech',
                            'Freelance / Creative', 'Healthcare / Shift Worker'],
    'chronotype'         : ['Intermediate', 'Night Owl', 'Morning Lark'],
    'primary_bedtime_app': ['News / Reading', 'Streaming (Netflix/Hulu)', 'Messaging / Chat',
                            'YouTube', 'Instagram / Reddit', 'TikTok / Reels'],
}

# Diputuskan manual (kunci ditulis dalam bentuk ternormalisasi)
PETA_MANUAL = {
    'gender': {'f': 'Female', 'm': 'Male'},
    # kolom lain tidak butuh keputusan manual
}

for k, valid in LABEL_BAKU.items():
    sebelum = df[k].nunique()                       # hitung SEBELUM transformasi
    kunci_k = kunci(df[k])

    # Otomatis: kunci ternormalisasi -> label baku. Manual: ditambahkan di atasnya.
    peta = {kunci(pd.Series([v]))[0]: v for v in valid}
    peta.update(PETA_MANUAL.get(k, {}))

    hasil = kunci_k.map(peta)

    # Pengaman: ada nilai non-missing yang tidak terpetakan?
    tak_terpetakan = set(kunci_k[kunci_k.notna() & hasil.isna()].unique())
    assert not tak_terpetakan, f'{k}: belum terpetakan -> {tak_terpetakan}'

    df[k] = hasil
    catat(f'Seragamkan {k}', sebelum - df[k].nunique(),
          'Spasi/huruf/pemisah oleh kode; singkatan diputuskan manual di PETA_MANUAL')
    print(k, '->', sorted(df[k].dropna().unique()))

  Seragamkan gender: 9 baris
gender -> ['Female', 'Male', 'Non-Binary']
  Seragamkan occupation_type: 10 baris
occupation_type -> ['Corporate 9-to-5', 'Freelance / Creative', 'Healthcare / Shift Worker', 'Remote Tech', 'Student']
  Seragamkan chronotype: 5 baris
chronotype -> ['Intermediate', 'Morning Lark', 'Night Owl']
  Seragamkan primary_bedtime_app: 12 baris
primary_bedtime_app -> ['Instagram / Reddit', 'Messaging / Chat', 'News / Reading', 'Streaming (Netflix/Hulu)', 'TikTok / Reels', 'YouTube']


In [16]:
# Penyeragaman kategori, dengan peta keputusan yang eksplisit
KATEGORI_VALID = {'Optimal Recovery', 'Mild Deficit', 'Moderate Debt', 'Severe Sleep Debt'}

# Hitung SEBELUM transformasi apa pun, agar log mencerminkan seluruh langkah
sebelum = df['sleep_debt_category'].nunique()

# Otomatis oleh kode: spasi dan huruf besar/kecil
df['sleep_debt_category'] = df['sleep_debt_category'].astype('string').str.strip().str.title()

# Diputuskan manual: salah ketik. Kunci ditulis dalam bentuk SETELAH title(),
# agar benar-benar cocok.
PETA_sleep_debt_category = {
    'Sever Sleep Debt': 'Severe Sleep Debt',
    'Moderat Debt'    : 'Moderate Debt',
    'Mild Defisit'    : 'Mild Deficit',
    'Optimal Recovry' : 'Optimal Recovery',
}
df['sleep_debt_category'] = df['sleep_debt_category'].replace(PETA_sleep_debt_category)

# Pengaman: pastikan tidak ada nilai di luar 4 kategori valid
tak_terpetakan = set(df['sleep_debt_category'].dropna().unique()) - KATEGORI_VALID
assert not tak_terpetakan, f'Masih ada kategori tak terpetakan: {tak_terpetakan}'

catat('Seragamkan sleep_debt_category', sebelum - df['sleep_debt_category'].nunique(),
      'Huruf dan spasi oleh kode; 4 salah ketik diputuskan manual lalu dicatat di peta')

print(sorted(df['sleep_debt_category'].dropna().unique()))

  Seragamkan sleep_debt_category: 4 baris
['Mild Deficit', 'Moderate Debt', 'Optimal Recovery', 'Severe Sleep Debt']


In [17]:
# Hasil Penyeragaman kategori
p5_kat      = df['sleep_debt_category']

print('P5_kat', p5_kat)


P5_kat 0       Severe Sleep Debt
1       Severe Sleep Debt
2       Severe Sleep Debt
3            Mild Deficit
4           Moderate Debt
              ...        
8595        Moderate Debt
8596    Severe Sleep Debt
8597         Mild Deficit
8598    Severe Sleep Debt
8599        Moderate Debt
Name: sleep_debt_category, Length: 8600, dtype: string


In [18]:
# Penyeragaman yes true, no false pada blue_light_filter_active
# CEK
print(df['blue_light_filter_active'].astype('string').str.strip().str.lower().value_counts(dropna=False))

# BERSIHKAN
sebelum = df['blue_light_filter_active'].astype('string').nunique()
PETA_BINER = {'1': 1, 'yes': 1, 'true': 1,
              '0': 0, 'no': 0, 'false': 0}

kunci_b = df['blue_light_filter_active'].astype('string').str.strip().str.lower()
hasil = kunci_b.map(PETA_BINER)

tak_terpetakan = set(kunci_b[kunci_b.notna() & hasil.isna()].unique())
assert not tak_terpetakan, f'Belum terpetakan: {tak_terpetakan}'

df['blue_light_filter_active'] = hasil.astype('Int64')   # Int64 agar NaN tetap boleh
catat('Seragamkan blue_light_filter_active', sebelum - df['blue_light_filter_active'].nunique(),
      'Yes/No/True/False/1/0 dipetakan ke 1/0')
print(df['blue_light_filter_active'].value_counts(dropna=False))

blue_light_filter_active
0        4215
1        3697
no        250
yes       200
true      125
false     113
Name: count, dtype: Int64
  Seragamkan blue_light_filter_active: 6 baris
blue_light_filter_active
0    4578
1    4022
Name: count, dtype: Int64


### **Exploratory Data**

In [19]:
# Memastikan bentuk data
df.head(10)

,user_id,age,gender,occupation_type,chronotype,bedtime_phone_minutes,primary_bedtime_app,screen_brightness_pct,blue_light_filter_active,caffeine_post_5pm_mg,physical_activity_min,sleep_latency_min,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,morning_alarm_snoozes,next_day_fatigue_score,sleep_debt_category
0,USR-00001,29,Female,Healthcare / Shift Worker,Intermediate,179.0,Instagram / Reddit,54.0,1,0.0,21,84.4,3.2,23.5,22.2,7,10.0,Severe Sleep Debt
1,USR-00002,58,Female,Student,Intermediate,163.0,YouTube,76.0,1,92.0,72,86.7,4.06,18.0,21.5,7,10.0,Severe Sleep Debt
2,USR-00003,41,Female,Healthcare / Shift Worker,Night Owl,100.0,TikTok / Reels,59.0,0,45.0,23,70.5,3.2,18.1,20.5,7,10.0,Severe Sleep Debt
3,USR-00004,36,Non-Binary,Remote Tech,Intermediate,34.0,YouTube,52.0,0,97.0,49,38.1,6.99,21.6,22.4,1,1.7,Mild Deficit
4,USR-00005,23,Female,Healthcare / Shift Worker,Intermediate,27.0,TikTok / Reels,82.0,1,0.0,38,32.3,4.58,24.1,<NA>,4,5.3,Moderate Debt
5,USR-00006,23,Male,Healthcare / Shift Worker,Intermediate,76.0,TikTok / Reels,39.0,0,136.0,42,50.6,5.87,21.4,17.5,4,5.0,Moderate Debt
6,USR-00007,20,Female,Remote Tech,Night Owl,53.0,TikTok / Reels,32.0,0,0.0,64,36.2,4.43,22.7,19.0,6,6.4,Moderate Debt
7,USR-00008,49,Male,Remote Tech,Morning Lark,113.0,YouTube,34.0,1,39.0,58,60.2,5.94,19.3,19.9,3,4.5,Moderate Debt
8,USR-00009,36,Male,Freelance / Creative,Morning Lark,65.0,News / Reading,35.0,1,0.0,25,28.2,8.0,22.2,18.5,1,1.0,Optimal Recovery
9,USR-00010,40,Female,Student,Night Owl,70.0,Streaming (Netflix/Hulu),63.0,1,115.0,59,53.7,4.7,23.5,16.6,4,7.3,Moderate Debt


In [20]:
# Melihat tipe dan kondisi data
df.info()
print("------------------------------------")
print("Dari tabel data di atas, dapat dilihat terdapat column yang \nmemiliki missing value seperti oocupation_type, chronotype dan lainnya")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8600 entries, 0 to 8599
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8600 non-null   string 
 1   age                       8600 non-null   Int64  
 2   gender                    8600 non-null   object 
 3   occupation_type           8429 non-null   object 
 4   chronotype                8473 non-null   object 
 5   bedtime_phone_minutes     8600 non-null   float64
 6   primary_bedtime_app       8600 non-null   object 
 7   screen_brightness_pct     8429 non-null   Float64
 8   blue_light_filter_active  8600 non-null   Int64  
 9   caffeine_post_5pm_mg      8429 non-null   Float64
 10  physical_activity_min     8600 non-null   Int64  
 11  sleep_latency_min         8344 non-null   Float64
 12  total_sleep_hours         8600 non-null   Float64
 13  deep_sleep_pct            8343 non-null   Float64
 14  rem_slee

In [21]:
# Cek dimensi data
df.shape


(8600, 18)

In [22]:
# Melihat penyebaran dan pemusatan data
df.describe()

,age,bedtime_phone_minutes,screen_brightness_pct,blue_light_filter_active,caffeine_post_5pm_mg,physical_activity_min,sleep_latency_min,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,morning_alarm_snoozes,next_day_fatigue_score
count,8600.0,8600.000000,8429.0,8600.0,8429.0,8600.0,8344.0,8600.0,8343.0,8539.0,8600.0,8600.0
mean,34.561395,59.204186,55.193855,0.467674,33.824534,35.713372,40.834348,6.270015,21.731823,19.104614,2.763605,3.789442
std,18.744652,36.868346,19.943725,0.498983,64.30334,20.802429,22.861151,1.499106,3.0704,2.546549,1.692847,2.680657
min,-5.0,1.000000,-20.0,0.0,-50.0,0.0,-10.0,-3.0,8.1,9.6,0.0,1.0
25%,25.0,31.000000,42.0,0.0,0.0,21.0,28.2,5.24,19.7,17.4,2.0,1.1
50%,32.0,51.000000,55.0,0.0,0.0,35.0,37.5,6.33,21.9,19.1,3.0,3.2
75%,42.0,79.000000,69.0,1.0,55.0,50.0,49.6,7.35,23.9,20.8,4.0,5.5
max,999.0,180.000000,150.0,1.0,2500.0,112.0,999.0,25.0,28.0,27.0,7.0,10.0


In [23]:
# seberapa bersih data ini?
# jumlah nilai kosong (missing) per kolom
missingValue = df.isnull().sum()
print(missingValue)
# jumlah baris duplikat
print("----------------------------------")
dataDuplikat = df.duplicated().sum()
print(dataDuplikat)

user_id                       0
age                           0
gender                        0
occupation_type             171
chronotype                  127
bedtime_phone_minutes         0
primary_bedtime_app           0
screen_brightness_pct       171
blue_light_filter_active      0
caffeine_post_5pm_mg        171
physical_activity_min         0
sleep_latency_min           256
total_sleep_hours             0
deep_sleep_pct              257
rem_sleep_pct                61
morning_alarm_snoozes         0
next_day_fatigue_score        0
sleep_debt_category           0
dtype: int64
----------------------------------
100


### **Cleaning Data**

In [24]:
df.drop(dataDuplikat)

,user_id,age,gender,occupation_type,chronotype,bedtime_phone_minutes,primary_bedtime_app,screen_brightness_pct,blue_light_filter_active,caffeine_post_5pm_mg,physical_activity_min,sleep_latency_min,total_sleep_hours,deep_sleep_pct,rem_sleep_pct,morning_alarm_snoozes,next_day_fatigue_score,sleep_debt_category
0,USR-00001,29,Female,Healthcare / Shift Worker,Intermediate,179.0,Instagram / Reddit,54.0,1,0.0,21,84.4,3.2,23.5,22.2,7,10.0,Severe Sleep Debt
1,USR-00002,58,Female,Student,Intermediate,163.0,YouTube,76.0,1,92.0,72,86.7,4.06,18.0,21.5,7,10.0,Severe Sleep Debt
2,USR-00003,41,Female,Healthcare / Shift Worker,Night Owl,100.0,TikTok / Reels,59.0,0,45.0,23,70.5,3.2,18.1,20.5,7,10.0,Severe Sleep Debt
3,USR-00004,36,Non-Binary,Remote Tech,Intermediate,34.0,YouTube,52.0,0,97.0,49,38.1,6.99,21.6,22.4,1,1.7,Mild Deficit
4,USR-00005,23,Female,Healthcare / Shift Worker,Intermediate,27.0,TikTok / Reels,82.0,1,0.0,38,32.3,4.58,24.1,<NA>,4,5.3,Moderate Debt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8595,USR-08496,61,Female,Remote Tech,Night Owl,18.0,Streaming (Netflix/Hulu),61.0,1,60.0,74,21.0,5.57,26.5,18.1,3,3.2,Moderate Debt
8596,USR-08497,64,Male,Student,Night Owl,138.0,TikTok / Reels,58.0,1,0.0,58,84.3,3.99,20.4,19.4,7,10.0,Severe Sleep Debt
8597,USR-08498,34,Male,Corporate 9-to-5,Intermediate,56.0,Streaming (Netflix/Hulu),56.0,0,94.0,33,40.6,7.58,17.4,20.7,2,3.7,Mild Deficit
8598,USR-08499,23,Male,Healthcare / Shift Worker,Night Owl,165.0,TikTok / Reels,72.0,0,0.0,52,123.3,3.2,14.2,15.8,7,10.0,Severe Sleep Debt


In [25]:
df.shape

(8600, 18)

In [26]:
# Definisikan aturan batas domain untuk setiap kolom
ATURAN = {
    'age': (0, 120),
    'bedtime_phone_minutes': (0, 1440),
    'screen_brightness_pct': (0, 100),
    'caffeine_post_5pm_mg': (0, 1000),
    'physical_activity_min': (0, 1440),
    'sleep_latency_min': (0, 1440),
    'total_sleep_hours': (0, 24),
    'deep_sleep_pct': (0, 100),
    'rem_sleep_pct': (0, 100),
    'morning_alarm_snoozes': (0, 100),
    'next_day_fatigue_score': (1, 10)
}

d = df  # Memetakan d ke DataFrame df agar kode di bawah berfungsi

rekap = {}
for k, (lo, hi) in ATURAN.items():
    if k in d.columns:
        m = (d[k] < lo) | (d[k] > hi)
        rekap[k] = int(m.sum())
        if m.any():
            print(f'{k:25s} {m.sum():3d} melanggar [{lo},{hi}] -> {sorted(d.loc[m, k].unique())}')
        d.loc[m, k] = np.nan

total = sum(rekap.values())
catat('Nilai di luar batas domain -> NaN', total, 'Aturan domain per kolom, bukan IQR')

if total == 0:
    print('PERINGATAN: 0 pelanggaran. Cek apakah sel ini sudah pernah dijalankan '
          'atau data sudah dibersihkan sebelumnya.')

age                         6 melanggar [0,120] -> [np.int64(-5), np.int64(999)]
screen_brightness_pct       6 melanggar [0,100] -> [np.float64(-20.0), np.float64(130.0), np.float64(150.0)]
caffeine_post_5pm_mg        4 melanggar [0,1000] -> [np.float64(-50.0), np.float64(2500.0)]
sleep_latency_min           2 melanggar [0,1440] -> [np.float64(-10.0)]
total_sleep_hours           6 melanggar [0,24] -> [np.float64(-3.0), np.float64(25.0)]
  Nilai di luar batas domain -> NaN: 24 baris


In [27]:
print("Jumlah missing values setelah pembersihan domain:")
display(df.isnull().sum())

Jumlah missing values setelah pembersihan domain:


,0
user_id,0
age,6
gender,0
occupation_type,171
chronotype,127
bedtime_phone_minutes,0
primary_bedtime_app,0
screen_brightness_pct,177
blue_light_filter_active,0
caffeine_post_5pm_mg,175


In [28]:
df['user_id'] = df['user_id'].str.strip()

print("Sebelum:", df.shape)
df = df.drop_duplicates()
df = df.reset_index(drop=True)
print("Sesudah:", df.shape)
print("user_id yang masih kembar:", df['user_id'].duplicated().sum())

Sebelum: (8600, 18)
Sesudah: (8500, 18)
user_id yang masih kembar: 0


**Mengubah angka yang tersimpan sebagai teks**

In [29]:
# Kolom 'age', 'bedtime_phone_minutes', dan 'rem_sleep_pct' sudah dibersihkan
# dan diubah menjadi tipe numerik di sel 'iTjN2R8E6HwC' sebelumnya.
# Kita hanya perlu menampilkan tipe data saat ini untuk memastikannya:

display(df[['age', 'bedtime_phone_minutes', 'rem_sleep_pct']].dtypes)
print("\nContoh data yang sudah dibersihkan:")
display(df[['age', 'bedtime_phone_minutes', 'rem_sleep_pct']].head())

,0
age,Int64
bedtime_phone_minutes,float64
rem_sleep_pct,Float64



Contoh data yang sudah dibersihkan:


,age,bedtime_phone_minutes,rem_sleep_pct
0,29,179.0,22.2
1,58,163.0,21.5
2,41,100.0,20.5
3,36,34.0,22.4
4,23,27.0,<NA>


**Menseragamkan penulisan kolom kategori**

In [30]:
kolom_teks = ['gender', 'occupation_type', 'chronotype', 'primary_bedtime_app', 'sleep_debt_category']

# Tahap 1: huruf kecil semua dan buang spasi di ujung
for kolom in kolom_teks:
    df[kolom] = df[kolom].str.strip().str.lower()

df['chronotype'] = df['chronotype'].str.replace('_', ' ')

# Tahap 2: arahkan semua variasi ke nama baku
df['gender'] = df['gender'].map({
    'female': 'Female', 'f': 'Female',
    'male': 'Male', 'm': 'Male',
    'non-binary': 'Non-Binary', 'non binary': 'Non-Binary'
})

df['occupation_type'] = df['occupation_type'].map({
    'corporate 9-to-5': 'Corporate 9-to-5',
    'remote tech': 'Remote Tech',
    'student': 'Student',
    'healthcare / shift worker': 'Healthcare / Shift Worker',
    'freelance / creative': 'Freelance / Creative'
})

df['chronotype'] = df['chronotype'].map({
    'intermediate': 'Intermediate',
    'night owl': 'Night Owl',
    'morning lark': 'Morning Lark'
})

df['primary_bedtime_app'] = df['primary_bedtime_app'].map({
    'tiktok / reels': 'TikTok / Reels',
    'youtube': 'YouTube',
    'instagram / reddit': 'Instagram / Reddit',
    'streaming (netflix/hulu)': 'Streaming (Netflix/Hulu)',
    'messaging / chat': 'Messaging / Chat',
    'news / reading': 'News / Reading'
})

# sekalian memperbaiki salah ketik
df['sleep_debt_category'] = df['sleep_debt_category'].map({
    'severe sleep debt': 'Severe Sleep Debt',
    'sever sleep debt': 'Severe Sleep Debt',
    'moderate debt': 'Moderate Debt',
    'moderat debt': 'Moderate Debt',
    'mild deficit': 'Mild Deficit',
    'mild defisit': 'Mild Deficit',
    'optimal recovery': 'Optimal Recovery',
    'optimal recovry': 'Optimal Recovery'
})

# Cek hasil: harusnya hanya nama baku (NaN = kosong dari awal)
for kolom in kolom_teks:
    print(df[kolom].value_counts(dropna=False))
    print()

gender
Female        4347
Male          3905
Non-Binary     248
Name: count, dtype: int64

occupation_type
Corporate 9-to-5             2781
Remote Tech                  2104
Student                      1584
Healthcare / Shift Worker     982
Freelance / Creative          879
NaN                           170
Name: count, dtype: int64

chronotype
Intermediate    3828
Night Owl       2410
Morning Lark    2135
NaN              127
Name: count, dtype: int64

primary_bedtime_app
TikTok / Reels              2198
YouTube                     1928
Instagram / Reddit          1630
Streaming (Netflix/Hulu)    1301
Messaging / Chat             847
News / Reading               596
Name: count, dtype: int64

sleep_debt_category
Moderate Debt        4462
Mild Deficit         2004
Optimal Recovery     1387
Severe Sleep Debt     647
Name: count, dtype: int64



**Menangani nilai yang tidak masuk akal (outlier)**

In [31]:
kolom_angka = ['age', 'bedtime_phone_minutes', 'screen_brightness_pct', 'caffeine_post_5pm_mg',
               'physical_activity_min', 'sleep_latency_min', 'total_sleep_hours',
               'deep_sleep_pct', 'rem_sleep_pct', 'morning_alarm_snoozes', 'next_day_fatigue_score']

# Lihat min dan max dulu
print(df[kolom_angka].describe().T[['min', 'max']])

# age (batas wajar 10 - 100)
salah = (df['age'] < 10) | (df['age'] > 100)
print("age tidak wajar:", salah.sum())
df.loc[salah, 'age'] = np.nan

# screen_brightness_pct (0 - 100)
salah = (df['screen_brightness_pct'] < 0) | (df['screen_brightness_pct'] > 100)
print("screen_brightness_pct tidak wajar:", salah.sum())
df.loc[salah, 'screen_brightness_pct'] = np.nan

# caffeine_post_5pm_mg (0 - 600)
salah = (df['caffeine_post_5pm_mg'] < 0) | (df['caffeine_post_5pm_mg'] > 600)
print("caffeine_post_5pm_mg tidak wajar:", salah.sum())
df.loc[salah, 'caffeine_post_5pm_mg'] = np.nan

# sleep_latency_min (0 - 300)
salah = (df['sleep_latency_min'] < 0) | (df['sleep_latency_min'] > 300)
print("sleep_latency_min tidak wajar:", salah.sum())
df.loc[salah, 'sleep_latency_min'] = np.nan

# total_sleep_hours (lebih dari 0 sampai 14)
salah = (df['total_sleep_hours'] <= 0) | (df['total_sleep_hours'] > 14)
print("total_sleep_hours tidak wajar:", salah.sum())
df.loc[salah, 'total_sleep_hours'] = np.nan

                         min    max
age                      0.0   65.0
bedtime_phone_minutes    1.0  180.0
screen_brightness_pct   10.0  100.0
caffeine_post_5pm_mg     0.0  250.0
physical_activity_min    0.0  112.0
sleep_latency_min        6.0  999.0
total_sleep_hours        0.0    9.8
deep_sleep_pct           8.1   28.0
rem_sleep_pct            9.6   27.0
morning_alarm_snoozes    0.0    7.0
next_day_fatigue_score   1.0   10.0
age tidak wajar: 2
screen_brightness_pct tidak wajar: 0
caffeine_post_5pm_mg tidak wajar: 0
sleep_latency_min tidak wajar: 2
total_sleep_hours tidak wajar: 2


**Menangani data kosong (missing value)**

In [32]:
print("Data kosong sebelum diisi:")
print(df.isna().sum())

# Kolom angka diisi median
kolom_isi_median = ['age', 'screen_brightness_pct', 'caffeine_post_5pm_mg', 'sleep_latency_min',
                    'total_sleep_hours', 'deep_sleep_pct', 'rem_sleep_pct']

for kolom in kolom_isi_median:
    nilai_median = df[kolom].median()
    df[kolom] = df[kolom].fillna(nilai_median)

# Kolom kategori diisi "Unknown"
df['occupation_type'] = df['occupation_type'].fillna('Unknown')
df['chronotype'] = df['chronotype'].fillna('Unknown')

# umur dijadikan bilangan bulat
df['age'] = df['age'].round().astype(int)

print("Sisa data kosong:", df.isna().sum().sum())

Data kosong sebelum diisi:
user_id                       0
age                           8
gender                        0
occupation_type             170
chronotype                  127
bedtime_phone_minutes         0
primary_bedtime_app           0
screen_brightness_pct       176
blue_light_filter_active      0
caffeine_post_5pm_mg        174
physical_activity_min         0
sleep_latency_min           259
total_sleep_hours             8
deep_sleep_pct              255
rem_sleep_pct                60
morning_alarm_snoozes         0
next_day_fatigue_score        0
sleep_debt_category           0
dtype: int64
Sisa data kosong: 0


**Output data bersih**

In [33]:
print("Ukuran data bersih:", df.shape)
print("Data kosong       :", df.isna().sum().sum())
print("Baris duplikat    :", df.duplicated().sum())
df.info()
print(df[kolom_angka].describe().T[['min', 'max', 'mean']])

df.to_csv('bedtime_screentime_sleep_debt_bersih.csv', index=False)
files.download('bedtime_screentime_sleep_debt_bersih.csv')

Ukuran data bersih: (8500, 18)
Data kosong       : 0
Baris duplikat    : 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8500 entries, 0 to 8499
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8500 non-null   string 
 1   age                       8500 non-null   int64  
 2   gender                    8500 non-null   object 
 3   occupation_type           8500 non-null   object 
 4   chronotype                8500 non-null   object 
 5   bedtime_phone_minutes     8500 non-null   float64
 6   primary_bedtime_app       8500 non-null   object 
 7   screen_brightness_pct     8500 non-null   Float64
 8   blue_light_filter_active  8500 non-null   Int64  
 9   caffeine_post_5pm_mg      8500 non-null   Float64
 10  physical_activity_min     8500 non-null   Int64  
 11  sleep_latency_min         8500 non-null   Float64
 12  total_sleep_hours         8500 non-null   F

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:

# 5. Jalankan pola F1-F5 (ukur k) — pakai fungsi cek_k yang sudah ada

def cek_k(data, kolom):
    ukuran_grup = data.groupby(kolom, observed=True).size()
    k = ukuran_grup.min()
    grup_berisi_1 = (ukuran_grup == 1).sum()
    total_grup = ukuran_grup.shape[0]
    return pd.Series({'k': k, 'grup_berisi_1': grup_berisi_1, 'total_grup': total_grup})

In [36]:
# F2 — sebelum penanganan (pakai df asli)
kolom_qi_sebelum = ['age', 'gender', 'occupation_type', 'chronotype']
hasil_sebelum_sleep = cek_k(df, kolom_qi_sebelum)
print("F2 — Sebelum:")
print(hasil_sebelum_sleep)

# Generalisasi: Membuat kelompok_umur dan aman_sleep terlebih dahulu agar tidak Error
aman_sleep = df.copy()
bins = [0, 25, 35, 45, 55, 100]
labels = ['<25', '25-34', '35-44', '45-54', '55+']
aman_sleep['kelompok_umur'] = pd.cut(aman_sleep['age'], bins=bins, labels=labels).astype(str)

# F3 — sesudah generalisasi
kolom_qi_sesudah = ['kelompok_umur', 'gender', 'occupation_type', 'chronotype']
hasil_sesudah_sleep = cek_k(aman_sleep, kolom_qi_sesudah)
print("\nF3 — Sesudah:")
print(hasil_sesudah_sleep)

F2 — Sebelum:
k                   1
grup_berisi_1     472
total_grup       1730
dtype: int64

F3 — Sesudah:
k                  1
grup_berisi_1     31
total_grup       284
dtype: int64
